# 02. Profiling: Geolocation Dataset
This notebook profiles `olist_geolocation_dataset.csv`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
print(f"Shape: {df.shape}\n")
print(df.dtypes)
df.head()

Shape: (1000163, 5)

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [2]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
pd.DataFrame({'count': missing, 'pct': missing_pct})

,count,pct
geolocation_zip_code_prefix,0,0.0
geolocation_lat,0,0.0
geolocation_lng,0,0.0
geolocation_city,0,0.0
geolocation_state,0,0.0


In [3]:
# Duplicates and Uniqueness
print(f"Full row duplicates: {df.duplicated().sum()}")
print(f"zip_code_prefix duplicates: {df.duplicated(subset=['geolocation_zip_code_prefix']).sum()}")
print(f"Row count == Distinct zip_code_prefix: {df.shape[0] == df['geolocation_zip_code_prefix'].nunique()}")

Full row duplicates: 261831
zip_code_prefix duplicates: 981148
Row count == Distinct zip_code_prefix: False


In [4]:
# Categorical Value Counts
print(df['geolocation_city'].value_counts().head())
print(df['geolocation_state'].value_counts().head())

geolocation_city
sao paulo         135800
rio de janeiro     62151
belo horizonte     27805
são paulo          24918
curitiba           16593
Name: count, dtype: int64
geolocation_state
SP    404268
MG    126336
RJ    121169
RS     61851
PR     57859
Name: count, dtype: int64


In [5]:
# Numeric Describe (lat/lng)
df[['geolocation_lat', 'geolocation_lng']].describe()

,geolocation_lat,geolocation_lng
count,1.000163e+06,1.000163e+06
mean,-2.117615e+01,-4.639054e+01
std,5.715866e+00,4.269748e+00
min,-3.660537e+01,-1.014668e+02
25%,-2.360355e+01,-4.857317e+01
50%,-2.291938e+01,-4.663788e+01
75%,-1.997962e+01,-4.376771e+01
max,4.506593e+01,1.211054e+02


In [6]:
# Specific checks: uniqueness of zip prefixes
n_unique_zips = df['geolocation_zip_code_prefix'].nunique()
print(f"Distinct zip code prefixes: {n_unique_zips}")
print(f"Rows per distinct zip code prefix (average): {df.shape[0] / n_unique_zips:.2f}")

# Top 5 zip codes by row count
df.groupby('geolocation_zip_code_prefix').size().sort_values(ascending=False).head()

Distinct zip code prefixes: 19015
Rows per distinct zip code prefix (average): 52.60


geolocation_zip_code_prefix
24220    1146
24230    1102
38400     965
35500     907
11680     879
dtype: int64

## Summary Findings
- **Completeness & Full Duplicates:** There are no missing values across the 1,000,163 rows. However, there are 261,831 fully duplicate rows (exact same coordinates and city strings for a given zip code).
- **Zip Codes are Not Unique:** The `geolocation_zip_code_prefix` is highly non-unique. There are only 19,015 distinct zip code prefixes, meaning there is an average of ~52.6 latitude/longitude entries per zip code prefix (the highest has over 1,100 points). 
- **Integration Strategy:** Before we can safely join this data to the `customers` or `sellers` tables on the zip code prefix (which expect a 1:1 match), we must collapse it. We plan to group by `geolocation_zip_code_prefix` and aggregate the coordinates (e.g., using the mean `lat` and `lng`) to create a true lookup table with 19,015 rows.